In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [3]:
# Create an embedding table for 10 date values
vocab_size = 10
embedding_dim = 16
input_embeddings = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)  # 16 is the embedding dimension
  # 3x16
# Example: Initialize the embedding table with random values
print(input_embeddings.weight)

Parameter containing:
tensor([[ 5.9641e-02, -8.3332e-01,  8.6113e-01, -3.0716e-01,  2.2472e+00,
         -1.6651e+00,  4.3291e-02,  1.0310e+00, -1.0785e+00, -1.1640e+00,
         -2.0434e+00, -1.5228e+00, -4.2671e-01,  8.3906e-01,  7.9920e-01,
         -3.5515e+00],
        [ 2.1611e+00, -1.2368e-01,  8.4337e-02,  2.4549e-01, -7.6609e-01,
          6.5155e-01, -4.1133e-01, -3.2646e-01,  1.0334e+00, -1.4368e+00,
         -1.5504e+00,  2.2014e-02,  2.6081e-01,  1.8600e-01, -9.8172e-01,
          2.6730e+00],
        [ 2.3001e-01,  3.5714e-01,  1.4003e-03,  6.9484e-01,  1.1733e+00,
         -5.9463e-01,  1.4949e+00,  4.7237e-01,  1.7875e-01,  8.5696e-01,
          2.7634e+00, -5.3696e-01,  1.4148e-01,  2.0537e+00,  1.7051e-01,
          1.8967e+00],
        [ 2.0143e+00, -1.0346e+00, -9.9021e-01, -2.9589e-01, -7.4703e-01,
          5.0741e-01, -1.1195e+00, -2.3875e-01,  1.6959e+00,  1.1256e+00,
         -1.2098e+00, -4.7657e-01,  4.9288e-01, -1.6858e+00,  7.7971e-01,
          9.9024e-01]

In [4]:
# Define input embeddings (example tensor)
embedding_dim=16
seq_len, batch_size, embed_dim = 5, 2, embedding_dim  # Use embedding_dim from the embedding table
input_embedding = input_embeddings.weight


# Define linear layers for key, query, and value projections
key_layer = nn.Linear(embed_dim, embed_dim)
query_layer = nn.Linear(embed_dim, embed_dim)
value_layer = nn.Linear(embed_dim, embed_dim)

# Generate key, query, and value tensors
keys = key_layer(input_embedding)
queries = query_layer(input_embedding)
values = value_layer(input_embedding)

print("Keys:", keys)
print("Queries:", queries)
print("Values:", values)

Keys: tensor([[-0.5224,  0.2390, -1.1894,  1.0594,  0.1486, -0.6411,  0.6984,  1.5274,
          0.8352,  1.1394,  0.4894,  0.1798,  2.1839,  0.2958, -0.1703, -0.7036],
        [-0.1254,  0.7069,  0.7661, -0.4789, -0.8341,  0.7608,  0.7243, -0.5163,
          0.5000,  0.7819,  0.0836,  0.7735, -0.5451, -0.9628, -0.6754,  1.1357],
        [ 0.9558,  0.7515, -0.3956, -0.7160,  0.6524, -0.3018,  1.7224, -0.4590,
         -0.3210, -1.5652, -0.6386, -0.8941,  0.1630, -0.0428, -0.0320,  0.4299],
        [-0.0329,  1.0037,  0.4513,  0.0366, -0.8784, -0.2533,  0.0853, -0.6643,
         -0.2823,  1.1622, -0.0626,  0.2412, -0.3128, -1.0838, -0.8888,  0.0446],
        [-0.8474, -0.2406,  0.7336,  0.3525,  0.1382,  0.6487,  0.7047,  0.2402,
         -0.3817,  0.8082, -0.1488,  0.8980,  0.7366,  0.2887, -0.2463, -0.0676],
        [-0.4709, -0.2198,  0.7564,  0.5685, -0.4682, -0.3668,  0.3798, -0.2116,
          0.0231,  1.6269,  0.4545,  1.3442,  0.4392, -0.6474, -0.1561,  0.8070],
        [-0.6545

In [ ]:

class CrossAttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super(CrossAttentionBlock, self).__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.dropout = dropout

        # Multi-head attention for cross-attention
        self.cross_attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)

        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

        # Layer normalization
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        # Dropout
        self.dropout_layer = nn.Dropout(dropout)

    def forward(self, query, key, value, attention_mask=None):
        """
        Forward pass for the cross-attention block.

        Args:
            query: Tensor of shape (seq_len_q, batch_size, embed_dim)
            key: Tensor of shape (seq_len_k, batch_size, embed_dim)
            value: Tensor of shape (seq_len_v, batch_size, embed_dim)
            attention_mask: Optional tensor for masking (seq_len_q, seq_len_k)

        Returns:
            Tensor of shape (seq_len_q, batch_size, embed_dim)
        """
        # Cross-attention
        attn_output, _ = self.cross_attention(query, key, value, attn_mask=attention_mask)
        query = query + self.dropout_layer(attn_output)
        print("query:", query)
        query = self.norm1(query)

        # Feed-forward network
        ffn_output = self.ffn(query)
        query = query + self.dropout_layer(ffn_output)
        query = self.norm2(query)

        return query

In [27]:
query = torch.rand(seq_len, batch_size, embed_dim)
key = torch.rand(seq_len, batch_size, embed_dim)
value = torch.rand(seq_len, batch_size, embed_dim)

In [ ]:
num_heads = 8
cross_attn_block = CrossAttentionBlock(embed_dim, num_heads)

In [43]:
#cheack the cross attention block
# Define the cross-attention block
num_heads = 8


# Generate a random tensor for the query, key, and value
# 5x2x16


# Apply the cross-attention block
# 5x2x16
output = cross_attn_block(query, key, value)
# print(output)


query: tensor([[[ 9.4006e-01,  1.4540e-01,  1.0371e-01,  5.9393e-01,  2.6242e-01,
           8.9661e-01,  9.4735e-01,  5.4258e-02,  9.4346e-02,  5.2265e-01,
           4.8180e-01,  8.0667e-01,  1.1876e-01,  4.6501e-01, -9.7864e-02,
           8.3117e-01],
         [ 4.2048e-01, -3.2929e-01,  1.6806e-01,  6.1163e-01, -1.3856e-01,
           3.0233e-01,  5.1500e-01,  1.7917e-01,  4.5513e-01,  7.8142e-01,
           5.1527e-01,  8.1186e-02,  6.8664e-01,  6.8313e-01,  6.5189e-01,
           1.4336e+00]],

        [[ 7.4560e-01,  4.4226e-01,  6.5385e-01,  3.2983e-01,  2.2763e-01,
           6.8080e-01,  8.2056e-01,  5.4219e-01,  3.2505e-01,  7.5864e-01,
           8.9466e-01,  1.0572e+00,  6.5683e-01,  1.1943e+00,  8.3265e-01,
           4.8285e-01],
         [ 9.7393e-01,  2.2305e-01,  3.8064e-01,  7.4917e-01,  6.0257e-01,
           3.2355e-01,  1.1979e+00,  7.1587e-01,  6.0065e-01,  8.2429e-01,
           3.7500e-01,  9.9192e-01,  6.7463e-01,  1.4743e-01, -7.1209e-02,
           9.3404e-

In [21]:
class Head(nn.Module):
    def __init__(self, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   
        q = self.query(x) 
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) 
        wei = F.softmax(wei, dim=-1) 
        wei = self.dropout(wei)
        v = self.value(x) 
        out = wei @ v 
        print(out)
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embd, block_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size, n_embd, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

num_heads = 8
cross_attn_block = CrossAttentionBlock(embed_dim, num_heads)

# Generate a random tensor for the query, key, and value
# 5x2x16
# Apply the cross-attention block
# 5x2x16
output = cross_attn_block(query, key, value)


    

query: tensor([[[ 1.5300,  1.1643,  0.8310,  0.9204,  0.4210,  0.5590,  0.8278,
           0.2540,  0.8551,  0.4231,  0.4723,  0.0118,  0.7671,  0.4583,
           0.7686,  0.4002],
         [ 0.9068,  0.6830,  0.4243,  1.1398, -0.1087,  0.4623,  0.9737,
           0.6340,  0.5561,  0.4841,  0.9100,  0.2892,  0.2232,  0.7992,
           0.4031,  0.8021]],

        [[ 1.7291,  0.3370,  0.3657,  1.3721,  0.2476,  0.4268,  0.8606,
           0.8349,  0.4823,  0.9672,  1.2576,  0.3157,  0.8187,  0.4596,
           0.9076,  0.5668],
         [ 1.5672,  0.8607,  0.4927,  1.2250,  0.7003,  0.5599,  1.4704,
           1.0383,  0.7149,  0.8548,  0.6905,  0.8244,  0.7308,  0.2993,
           1.1774,  0.5001]],

        [[ 1.5859,  0.5784,  0.3437,  1.2271,  0.7161,  0.4449,  0.0733,
           0.9206,  1.0908,  0.6326,  0.6770,  0.0366,  0.3440,  0.6649,
           0.6683, -0.0708],
         [ 0.1122,  0.5063,  0.7153,  0.6823,  0.4681,  0.4951,  1.3991,
           0.4369,  0.1065,  0.3055,  0.5

In [7]:
class Transformer(nn.Module):
    def __init__(self, embed_dim, num_heads, num_layers, dropout=0.1):
        super(Transformer, self).__init__()
        self.layers = nn.ModuleList([
            CrossAttentionBlock(embed_dim, num_heads, dropout) for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, query, key, value, attention_mask=None):
        """
        Forward pass for the transformer.

        Args:
            query: Tensor of shape (seq_len_q, batch_size, embed_dim)
            key: Tensor of shape (seq_len_k, batch_size, embed_dim)
            value: Tensor of shape (seq_len_v, batch_size, embed_dim)
            attention_mask: Optional tensor for masking (seq_len_q, seq_len_k)

        Returns:
            Tensor of shape (seq_len_q, batch_size, embed_dim)
        """
        for layer in self.layers:
            query = layer(query, key, value, attention_mask)
        return self.norm(query)

# Example usage
num_layers = 6
transformer = Transformer(embed_dim, num_heads, num_layers)

# Apply the transformer
output = transformer(query, key, value)
print(output)

tensor([[[ 1.0509,  0.0772, -0.2065, -1.0314, -0.1892,  2.2188, -0.9984,
          -1.8543,  0.0914,  1.3654, -0.3767, -0.5469, -0.3379, -0.8137,
           0.5569,  0.9945],
         [ 0.4365,  0.1993,  1.9317,  0.5159, -0.0170, -0.0251, -1.6328,
          -1.5888, -0.1249,  1.2722,  0.6275, -1.2495, -0.9551, -0.8608,
           0.3774,  1.0935]],

        [[ 0.1524, -0.1770,  0.4670,  0.2818,  0.4135,  1.5734, -1.4746,
          -1.0666, -1.2459,  1.2143,  0.6445, -1.2602, -1.5423, -0.0444,
           0.7598,  1.3042],
         [ 1.4767,  0.0567,  1.5046, -0.4591, -1.2135,  1.9122, -0.1693,
          -1.7180,  0.1116,  0.0967, -0.0547, -1.3904, -0.6196, -0.5126,
           0.2173,  0.7616]],

        [[ 0.4805,  0.0783,  1.8056, -0.3454,  0.9345,  1.4039, -0.3410,
          -1.2664, -0.6997,  1.4799, -0.9570, -1.7354,  0.4347, -0.3368,
          -0.9537,  0.0179],
         [ 1.2602,  0.5565,  0.8831,  0.2675, -0.1380,  1.3620, -1.5817,
          -0.4864, -1.4999,  1.2374, -0.1235, -1